## Collaborative filtering implementation using textual desc. of movies

`tags` column created using `overview`, `genres`, `tagline`.
<br/>
If any row contains null value in any of the 3 columns are filled with '' (empty space).

In [1]:
import numpy as np
import pandas as pd

### Loading dataset ...

In [2]:
df = pd.read_csv("../dataset/updated_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42278 entries, 0 to 42277
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         42277 non-null  object 
 1   overview      41363 non-null  object 
 2   genres        39996 non-null  object 
 3   tagline       19052 non-null  object 
 4   vote_average  42277 non-null  float64
 5   popularity    42277 non-null  float64
 6   tags          42278 non-null  object 
dtypes: float64(2), object(5)
memory usage: 2.3+ MB


We'll apply Collaborative filtering based on newly created `tags` column, for that first we've to preprocess the dataset like removing punctuation marks, stop words, lemmatization and atlast tokenization.

In [3]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\papsr\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\papsr\AppData\Roaming\nltk_data...


True

In [4]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [6]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z\s]', '', text)

    # tokenization
    tokens = text.split(' ')

    # first remove all stop words and then perform lemmatization
    tokens = [
        word for word in tokens if word not in stop_words
    ]
    tokens = [
        lemmatizer.lemmatize(word) for word in tokens
    ]

    return " ".join(tokens)

In [8]:
df['tags'] = df['tags'].apply( lambda x: preprocess_text(x) )

before going further, `reset_index` to make sure all indices arranges in serial order.

In [10]:
df = df.reset_index(drop=True)

### Separating `indices`

be'coz later we required this when finding movie indices.

In [11]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
tfidf = TfidfVectorizer(stop_words='english',
                        max_features=50000,     # vocab size
                        ngram_range=(1, 2),)

In [14]:
tfidf_matrix = tfidf.fit_transform(df['tags'])
tfidf_matrix.shape

(42278, 50000)

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
def recommend(title, n=10):
    if title not in indices:
        return ['movie not found']

    idx = indices[title]

    # calculate cosine similarity of current movie with all the other movies.
    similarity_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # argsort, sorts on the basis of index instead of values
    # reverse the sorted list and select top `n` indices.
    similar_index = similarity_score.argsort()[::-1][1:n+1]

    return df['title'].iloc[similar_index]

In [28]:
recommend('Toy Story')

2963                Toy Story 2
14771               Toy Story 3
23226                 Small Fry
27250           Superstar Goofy
16486                 Group Sex
6345     What's Up, Tiger Lily?
11048    For Your Consideration
1926                  Condorman
1069      Rebel Without a Cause
485                      Malice
Name: title, dtype: object

## HW: Recommend only those movies, the which user has not watched

In [34]:
df['genres'] = df['genres'].fillna('')
df['overview'] = df['overview'].fillna('')

In [35]:
def precision_at_k(title, k=10):
    recs = recommend(title, k)
    target_genres = set(df.loc[indices[title], 'genres'].split())

    hits = 0
    for r in recs:
        r_genres = set(df.loc[indices[r], 'genres'].split())
        if target_genres & r_genres:
            hits += 1

    return hits / k

In [38]:
scores = [precision_at_k(t) for t in df['title'].sample(200)]
np.mean(scores)

0.6739999999999999

In [40]:
def recall_at_k(title, k=10):
    if title not in indices:
        return None

    target_genres = set(df.loc[indices[title], 'genres'].split())
    if not target_genres:
        return None   # recall undefined

    relevant = df[
        df['genres'].apply(
            lambda g: bool(set(g.split()) & target_genres)
        )
    ]

    if len(relevant) == 0:
        return None

    recs = recommend(title, k)
    hits = sum(r in relevant['title'].values for r in recs)

    return hits / len(relevant)

In [41]:
scores = [
    recall_at_k(t)
    for t in df['title'].sample(200)
]

scores = [s for s in scores if s is not None]
np.mean(scores)

0.0006827741685932576

In [44]:
def mrr_at_k(title, k=10):
    if title not in indices:
        return None

    recs = recommend(title, k)
    target_genres = set(df.loc[indices[title], 'genres'].split())

    if not target_genres:
        return None

    for rank, r in enumerate(recs, start=1):
        r_genres = set(df.loc[indices[r], 'genres'].split())
        if target_genres & r_genres:
            return 1 / rank

    return 0

In [45]:
scores = [
    mrr_at_k(t)
    for t in df['title'].sample(200)
]

scores = [s for s in scores if s is not None]
np.mean(scores)

0.8064562862181908

In [43]:
import time

start = time.time()
recommend("Jumanji")
time.time() - start

0.036130666732788086

In [48]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[['popularity_norm', 'vote_norm']] = scaler.fit_transform(
    df[['popularity', 'vote_average']]
)

HybridScore=α⋅ContentSimilarity+β⋅Popularity+γ⋅Rating

In [46]:
def hybrid_recommend(title, k=10, alpha=0.7, beta=0.2, gamma=0.1):
    if title not in indices:
        return ['movie not found']

    idx = indices[title]

    # Content similarity
    sim_scores = cosine_similarity(
        tfidf_matrix[idx], tfidf_matrix
    ).flatten()

    # Build candidate frame
    candidates = df.copy()
    candidates['similarity'] = sim_scores

    # Remove self-match
    candidates = candidates[candidates.index != idx]

    # Hybrid score
    candidates['hybrid_score'] = (
        alpha * candidates['similarity']
        + beta * candidates['popularity_norm']
        + gamma * candidates['vote_norm']
    )

    return (
        candidates
        .sort_values('hybrid_score', ascending=False)
        .head(k)['title']
    )

In [49]:
def hybrid_precision_at_k(title, k=10):
    recs = hybrid_recommend(title, k)
    target_genres = set(df.loc[indices[title], 'genres'].split())

    hits = 0
    valid = 0

    for r in recs:
        r_genres = set(df.loc[indices[r], 'genres'].split())
        if not r_genres:
            continue
        valid += 1
        if target_genres & r_genres:
            hits += 1

    return hits / valid if valid > 0 else 0

In [50]:
titles = df['title'].sample(100)

cb_scores = [precision_at_k(t) for t in titles]
hyb_scores = [hybrid_precision_at_k(t) for t in titles]

np.mean(cb_scores), np.mean(hyb_scores)

(0.627, 0.6233611111111111)

In [55]:
def hybrid_mmr_precision_at_k(title, k=10):
    if title not in indices:
        return None

    recs = hybrid_mmr_at_k(title, k)
    target_genres = set(df.loc[indices[title], 'genres'].split())

    if not target_genres:
        return None

    hits, valid = 0, 0

    for r in recs:
        r_genres = set(df.loc[indices[r], 'genres'].split())
        if not r_genres:
            continue
        valid += 1
        if target_genres & r_genres:
            hits += 1

    return hits / valid if valid else None

In [56]:
scores = [
    hybrid_mmr_precision_at_k(t)
    for t in df['title'].sample(200)
]

scores = [s for s in scores if s is not None]
np.mean(scores)

0.5964349376114082

In [51]:
recommend("Jumanji")

33276             Aschenputtel
9258                 Word Wars
20608             Table No. 21
28404          The Games Maker
34977       The Ouija Exorcism
18902    Indie Game: The Movie
8591                   Quintet
14925          Le Pont du Nord
13229               Rhinoceros
41301        Liar Game: Reborn
Name: title, dtype: object

In [52]:
hybrid_recommend("Jumanji")

28824                           Minions
33276                      Aschenputtel
18902             Indie Game: The Movie
14925                   Le Pont du Nord
13229                        Rhinoceros
9258                          Word Wars
25068    Guardians of the Galaxy Vol. 2
20608                      Table No. 21
28404                   The Games Maker
8591                            Quintet
Name: title, dtype: object